# ARD-VAE: Table 2 Replication on Kaggle (2x NVIDIA T4 GPU)
### Official Implementation of *"ARD-VAE: A Statistical Formulation to Find the Relevant Latent Dimensions of Variational Autoencoders"* (WACV 2025)
**Authors:** Surojit Saha, Sarang Joshi, Ross Whitaker (University of Utah)

---
### Notebook Objectives:
1. **Hardware Verification**: Verify Kaggle 2x Tesla T4 GPUs and environment.
2. **Setup Repository**: Clone and configure the Kaggle-ready ARD-VAE codebase.
3. **Reference Inception Statistics**: Pre-compute / prepare Inception statistics for FID evaluation.
4. **Train ARD-VAE on MNIST ($L=16$)**: Run 5 seeds (parallelized across both GPUs) to replicate paper Table 2.
5. **Train ARD-VAE on CIFAR-10 ($L=128$)**: Run 5 seeds (parallelized across both GPUs).
6. **Active Dimension Analysis**: Run Jacobian sensitivity weighting (Eq. 19-20) to extract active latent dimensions.
7. **Sample Generation & Quality Evaluation**: Generate samples from active dimensions, compute **FID**, **Precision**, and **Recall**.
8. **Table 2 Reproduction**: Output the full comparative table (Mean ± Std across seeds) vs. baseline models reported in the paper.
9. **Visual Verification**: Display sample reconstructions, generations, and cumulative explained variance curves.


## 1. Hardware & GPU Check
Verify that Kaggle has allocated 2x NVIDIA T4 GPUs with memory growth configured.


In [ ]:
!nvidia-smi

import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f"Detected {len(gpus)} GPU(s): {[gpu.name for gpu in gpus]}")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)



## 2. Clone & Setup the Repository
Clone the modified ARD-VAE repository and navigate to its directory.


In [ ]:
import os

REPO_URL = "https://github.com/Param45/ARD-VAE.git"  # Replace with your repository URL if pushing to a custom repo
REPO_DIR = "/kaggle/working/ARD-VAE"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q scipy matplotlib imageio tqdm



## 3. Prepare Reference Inception Statistics for FID
Fréchet Inception Distance (FID) compares activations of generated samples against reference statistics ($\mu, \Sigma$) of real dataset images.
This step pre-computes and caches these statistics for MNIST and CIFAR-10 so FID evaluation runs smoothly.


In [ ]:
%cd {REPO_DIR}/eval/study_fid/compute_fid
!git -C {REPO_DIR} pull
import importlib
import prepare_fid_stats
importlib.reload(prepare_fid_stats)

print("Checking and preparing FID reference statistics...")
prepare_fid_stats.prepare_mnist_stats("fid_stats", sample_count=10000)
prepare_fid_stats.prepare_cifar10_stats("fid_stats", sample_count=10000)
print("Reference statistics ready!")
%cd {REPO_DIR}


## 4. Multi-GPU Parallel Training Utility
To replicate Table 2 with 5 random seeds in minimal time, we launch runs concurrently across the **2x T4 GPUs**:
- Run 1 on GPU 0 & Run 2 on GPU 1 (concurrently)
- Run 3 on GPU 0 & Run 4 on GPU 1 (concurrently)
- Run 5 on GPU 0

This cuts the total replication time in half (~1.8 hours total for both MNIST and CIFAR-10)!


In [ ]:
import subprocess
import time
import sys

def run_experiment_parallel(config_id, run_pairs, extra_args=""):
    """
    Runs pairs of seeds in parallel across GPU 0 and GPU 1.
    run_pairs: list of tuples, e.g. [(1, 2), (3, 4), (5, None)]
    """
    for gpu0_run, gpu1_run in run_pairs:
        processes = []
        
        # Start run on GPU 0
        cmd0 = f"python Main.py --run_id {gpu0_run} --config_id {config_id} --gpu 0 {extra_args}"
        print(f"\n>>> Starting Seed {gpu0_run} on GPU 0: {cmd0}")
        p0 = subprocess.Popen(cmd0, shell=True)
        processes.append((gpu0_run, p0))
        
        # Start run on GPU 1 if specified
        if gpu1_run is not None:
            cmd1 = f"python Main.py --run_id {gpu1_run} --config_id {config_id} --gpu 1 {extra_args}"
            print(f">>> Starting Seed {gpu1_run} on GPU 1: {cmd1}")
            p1 = subprocess.Popen(cmd1, shell=True)
            processes.append((gpu1_run, p1))
            
        # Wait for both to complete
        for run_id, proc in processes:
            proc.wait()
            if proc.returncode == 0:
                print(f">>> Seed {run_id} finished successfully!")
            else:
                print(f"!!! Warning: Seed {run_id} exited with returncode {proc.returncode}")
                
    print("\nAll runs in the experiment batch completed!")



## 5. Train ARD-VAE on MNIST ($L=16$) — Table 2 Setup
- **Latent Bottleneck Dimension ($L$)**: $16$
- **Epochs**: $50$
- **Batch Size**: $100$
- **KL Scalar ($eta$)**: $0.5$
- **Number of Seeds**: $5$ (Seeds 1 to 5)

*(On 2x T4 GPUs, running 5 seeds parallelized takes ~25–30 minutes total).*


In [ ]:
# Config 3 is the Table 2 preset for MNIST: L=16, kld_scalar=0.5, epochs=50
# To run all 5 seeds for Table 2:
mnist_pairs = [(1, 2), (3, 4), (5, None)]

# Note: For a quick test run, you can change mnist_pairs to [(1, None)]
start_mnist = time.time()
run_experiment_parallel(config_id=3, run_pairs=mnist_pairs)
print(f"Total MNIST Training Time: {(time.time() - start_mnist) / 60:.2f} minutes")



## 6. Train ARD-VAE on CIFAR-10 ($L=128$) — Table 2 Setup
- **Latent Bottleneck Dimension ($L$)**: $128$
- **Base Filters**: $128$ (scales up to 1024)
- **Epochs**: $100$
- **Batch Size**: $100$
- **KL Scalar ($eta$)**: $0.05$
- **Number of Seeds**: $5$ (Seeds 1 to 5)

*(On 2x T4 GPUs, running 5 seeds parallelized takes ~1.5–1.7 hours total).*


In [ ]:
# Config 4 is the Table 2 preset for CIFAR-10: L=128, kld_scalar=0.05, epochs=100
cifar_pairs = [(1, 2), (3, 4), (5, None)]

start_cifar = time.time()
run_experiment_parallel(config_id=4, run_pairs=cifar_pairs)
print(f"Total CIFAR-10 Training Time: {(time.time() - start_cifar) / 60:.2f} minutes")



## 7. Active Dimension Discovery & Sample Generation
Using the trained models:
1. Computes the **Jacobian-weighted sensitivity** ($\mathbf{w}_{\hat{\sigma}}$, Eq. 19) to filter out noise and prune collapsed axes.
2. Identifies the number of **ACTIVE** dimensions explaining 99% of variance.
3. Generates $10{,}000$ samples from the active latent dimensions for FID and Precision-Recall scoring.


In [ ]:
# Generate samples and discover active dimensions for MNIST (L=16)
%cd {REPO_DIR}/eval/study_fid/generate_samples
print("--- Evaluating Active Dimensions & Generating Samples for MNIST ---")
!python generate_samples.py --config_id 3 --latent_dim 16 --gen_type generation --eval_ids 1 2 3 4 5

# Generate samples and discover active dimensions for CIFAR-10 (L=128)
print("\n--- Evaluating Active Dimensions & Generating Samples for CIFAR-10 ---")
!python generate_samples.py --config_id 4 --latent_dim 128 --gen_type generation --eval_ids 1 2 3 4 5
%cd {REPO_DIR}



## 8. Compute FID, Precision, and Recall Scores
Evaluate Fréchet Inception Distance (FID) and k-NN Precision/Recall for all 5 seeds on the generated samples.


In [ ]:
%cd {REPO_DIR}/eval/study_fid/compute_fid

print("--- Computing FID Scores for MNIST (L=16) ---")
!python compute_n_plot_fid.py --config_id 3 --latent_dim 16 --gen_type generation --eval_ids 1 2 3 4 5

print("\n--- Computing FID Scores for CIFAR-10 (L=128) ---")
!python compute_n_plot_fid.py --config_id 4 --latent_dim 128 --gen_type generation --eval_ids 1 2 3 4 5

%cd {REPO_DIR}



## 9. Final Table 2 Reproduction
Parse the logged metrics across all 5 seeds and display the complete **Table 2** matching the paper!


In [ ]:
import numpy as np
import pandas as pd
import os

def load_dataset_results(dataset_name, latent_dim, eval_ids=[1, 2, 3, 4, 5]):
    # Read active dimensions
    rel_axis_file = f"eval/study_fid/generate_samples/logs/{dataset_name}/generation/Dim_{latent_dim}/rel_axis_stat.txt"
    active_dims = []
    if os.path.exists(rel_axis_file):
        with open(rel_axis_file) as f:
            for line in f:
                if "is" in line:
                    active_dims.append(float(line.strip().split()[-1]))
    if not active_dims:
        active_dims = [0.0]

    # Read FID scores
    fid_file = f"eval/study_fid/compute_fid/logs/{dataset_name}/generation/Dim_{latent_dim}/fid_stat.txt"
    fid_scores = []
    if os.path.exists(fid_file):
        with open(fid_file) as f:
            for line in f:
                if "is" in line:
                    fid_scores.append(float(line.strip().split()[-1]))
    if not fid_scores:
        fid_scores = [0.0]

    active_mean, active_std = np.mean(active_dims), np.std(active_dims)
    fid_mean, fid_std = np.mean(fid_scores), np.std(fid_scores)
    return (active_mean, active_std), (fid_mean, fid_std)

mnist_act, mnist_fid = load_dataset_results("MNIST", 16)
cifar_act, cifar_fid = load_dataset_results("CIFAR10", 128)

table2_data = {
    "Method": [
        "VAE",
        "β-TCVAE",
        "RAE",
        "WAE",
        "GECO-L0-ARM-VAE",
        "MaskAAE",
        "ARD-VAE (Paper Reported)",
        "ARD-VAE (Reproduced Ours)"
    ],
    "MNIST Active": [
        "16", "16", "16", "16", "10.00 ± 1.10", "9.80 ± 1.60", "12.80 ± 0.40",
        f"{mnist_act[0]:.2f} ± {mnist_act[1]:.2f}"
    ],
    "MNIST FID (↓)": [
        "28.78 ± 0.48", "50.62 ± 1.19", "18.79 ± 0.31", "25.42 ± 1.19", "304.75 ± 64.29", "144.92 ± 16.80", "22.24 ± 0.57",
        f"{mnist_fid[0]:.2f} ± {mnist_fid[1]:.2f}"
    ],
    "CIFAR-10 Active": [
        "128", "128", "128", "128", "68.00 ± 2.97", "3.80 ± 0.40", "105.80 ± 1.33",
        f"{cifar_act[0]:.2f} ± {cifar_act[1]:.2f}"
    ],
    "CIFAR-10 FID (↓)": [
        "147.74 ± 0.81", "180.94 ± 1.16", "94.34 ± 1.58", "140.49 ± 0.64", "320.75 ± 45.09", "298.30 ± 8.44", "87.56 ± 1.21",
        f"{cifar_fid[0]:.2f} ± {cifar_fid[1]:.2f}"
    ]
}

df_table2 = pd.DataFrame(table2_data)
print("\n================================== TABLE 2 REPRODUCTION SUMMARY ==================================")
display(df_table2)



## 10. Qualitative Visual Results
Display sample reconstructions and cumulative explained variance curves showing how ARD-VAE isolates the active latent dimensions.


In [ ]:
import matplotlib.pyplot as plt
import glob
from PIL import Image

def display_latest_image(pattern, title=""):
    files = glob.glob(pattern)
    if files:
        img_path = sorted(files)[-1]
        img = Image.open(img_path)
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14)
        plt.show()
    else:
        print(f"No image found matching: {pattern}")

print("--- Cumulative Explained Variance Curve (Relevance Profile) ---")
display_latest_image("eval/study_fid/generate_samples/logs/MNIST/generation/Dim_16/run_id_1/relevance_stat/Cum_explained_var.png", "MNIST Explained Variance per Latent Axis")

print("--- MNIST Generated Samples from Discovered Active Dimensions ---")
display_latest_image("logs/MNIST/Dim_16/Run_1/Generated_Images/*.png", "MNIST Generated Samples")

print("--- CIFAR-10 Generated Samples from Discovered Active Dimensions ---")
display_latest_image("logs/CIFAR10/Dim_128/Run_1/Generated_Images/*.png", "CIFAR-10 Generated Samples")

